In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.special import gamma
from numba import jit
from regression_common import ridge_pair, lasso_coefficients, sparse_indices, discrepancy_candidate
from regression_results import BootstrapRun
from regression_spectral import SpectralDesign
from regression_bootstrap import bootstrap_errors, validate_backend
from threadpoolctl import threadpool_limits


In [ ]:
def compute_sigma(Y, X, beta):
    sigma = np.linalg.norm(Y - X @ beta)
    return sigma / np.sqrt(Y.shape[0])


In [ ]:
@jit(nopython=True)
def landweber(X, Y, t, max_iter, beta_reference):
    beta_cur = np.zeros(X.shape[1])
    completed_steps = 0
    target_steps = max_iter
    turning_found = False
    for k in range(1, max_iter + 1):
        beta_prev = beta_cur
        X_beta_prev = np.dot(X, beta_prev)
        beta_cur = beta_prev + t * np.dot(X.T, (Y - X_beta_prev))
        # if np.linalg.norm(Y - np.dot(X, beta_cur)) <= delta:
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta_reference) >= np.linalg.norm(beta_prev - beta_reference):
                target_steps = min(max_iter, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps

@jit(nopython=True)
def landweber_debias(X, Y, t, max_iter):
    beta_cur = np.zeros(X.shape[1])
    for k in range(1, max_iter+1):
        beta_prev = beta_cur
        X_beta_prev = np.dot(X, beta_prev)
        beta_cur = beta_prev + t * np.dot(X.T, (Y - X_beta_prev))
    return beta_cur


In [ ]:


def compute_norminf_Landweber(seed):
    rng = np.random.default_rng(seed)
    ordinary_response = X @ beta_Landweber + rng.normal(0, sigma_Landweber, n)
    debiased_response = X @ beta_Landweber_debias + rng.normal(0, sigma_Landweber_debias, n)
    # Algorithm 1 keeps the original sample's selected parameter fixed.
    ordinary_star = landweber_debias(X, ordinary_response, t1, k_Landweber)
    debiased_star = landweber_debias(X, debiased_response, t1, 2 * k_Landweber)
    ordinary_star[np.abs(ordinary_star) <= bn_Landweber] = 0
    debiased_star[np.abs(debiased_star) <= bn_Landweber_debias] = 0
    ordinary_error = np.linalg.norm(ordinary_star - beta_Landweber, ord=np.inf)
    debiased_error = np.linalg.norm(debiased_star - beta_Landweber_debias, ord=np.inf)
    return ordinary_error, debiased_error


In [ ]:
beta_names = [
    'beta_Landweber', 'beta_Landweber_debias',
]

variable_names = [
    'Landweber', 'Landweber_debias',
]


In [ ]:
n        = 1000
p        = 1000
I        = np.eye(p)
t1       = 2.5e-6
max_iter = 10000
k_max    = 1000
tau      = 0.9

i_landweber = 0
i_landweber_debias = 0
Sigma_landweber = 0
Sigma_landweber_debias = 0
Inf_norm_landweber = 0
Inf_norm_landweber_debias = 0
Norm_landweber = 0
Norm_landweber_debias = 0

# Output controls do not change the estimators or parameter selection.
N = 1000
B = 500
confidence = 0.95
noise_sd = 2.0
seed_root = 20261006
output_root = 'outputs/regression-bootstrap-v2'
resume_from = None

# Execution controls only; statistical settings remain unchanged.
backend = 'cpu'
batch_size = 64
cpu_workers = 4
blas_threads = 8
gpu_devices = (0,)
backend_info = validate_backend(backend, gpu_devices=gpu_devices)


In [ ]:
# Keep the seeded covariance SVD independent of execution threads.
with threadpool_limits(limits=1):
    np.random.seed(24)
    cov_matrix = np.full((p, p), 0.5)
    np.fill_diagonal(cov_matrix, 2.0)

    mean_vector = np.zeros(p)
    X = np.random.multivariate_normal(mean_vector, cov_matrix, n)

    U, S, Vt = np.linalg.svd(X)

    beta = np.zeros(p)


    beta[0:5]   = 2.0   # βi = 2.0, i = 1, 2, 3, 4, 5
    beta[5:10]  = -2.0  # βi = -2.0, i = 6, 7, 8, 9, 10
    beta[10:15] = 1.0   # βi = 1.0, i = 11, 12, 13, 14, 15
    beta[15:20] = -1.0  # βi = -1.0, i = 16, 17, 18, 19, 20

    # Generate 20 random indices
    indices = sparse_indices(np.random, p, size=20)

    # Get the indices of non-zero elements in beta
    nonzero_indices = np.nonzero(beta)[0]
    random_beta = np.zeros(p)
    random_beta[indices] = beta[nonzero_indices]
    beta = random_beta
    spectral_design = SpectralDesign(X, svd=(U, S, Vt))


In [ ]:
run_config = {
    'method': 'Landweber', 'n': n, 'p': p, 'N': N, 'B': B,
    'sigma': noise_sd, 'confidence': confidence, 'seed_root': seed_root,
    'design_seed': 24, 'noise_model': 'normal',
    'largest_singular_value': float(S[0]),
    'execution': {'backend': backend, 'batch_size': batch_size,
                  'cpu_workers': cpu_workers, 'blas_threads': blas_threads,
                  'gpu_devices': list(gpu_devices)},
    'backend_info': backend_info,
    'preprocess_blas_threads': 1,
    'threshold_grid': [0.001, 0.001, 1.0],
    'threshold_percentile': 72.5,
    'method_settings': {'selection': 'AOSR', 'step': t1, 'max_steps': max_iter, 'minimum_steps': 500},
}
with threadpool_limits(limits=blas_threads), BootstrapRun(run_config, X, beta, 'Bootstraplandweber.ipynb',
                  output_root=output_root, resume_from=resume_from) as run:
    print('Run directory:', run.directory)
    for x in range(run.start_index, N):
        print(x)
        outer_rng = np.random.default_rng((seed_root, x, 0, 0))


        mean    = 0
        std_dev = noise_sd


        e = outer_rng.normal(mean, std_dev, n)
        norm_e = np.linalg.norm(e)
        Y = np.dot(X, beta) + e

        selection_settings = dict(run_config['method_settings'])
        delta = tau * norm_e
        beta_Landweber, beta_Landweber_debias, selected_parameter, selection_diagnostics = spectral_design.select(
            Y, 'Landweber', selection_settings, beta)
        k_Landweber = int(selected_parameter)
        selected_residual_norm = selection_diagnostics['residual_norm']
        sigma_Landweber = compute_sigma(Y, X, beta_Landweber)
        sigma_Landweber_debias = compute_sigma(Y, X, beta_Landweber_debias)

        result = []

        for i in range(1, 1001):
            bn = 0.001 * i

            norms = [bn]
            for var_name in beta_names:
                var_copy = globals()[var_name].copy()
                var_copy[np.abs(var_copy) <= bn] = 0
                norm = np.linalg.norm(var_copy - beta)
                norms.append(norm)
            result.append(norms)
        columns = ['bn'] + [f'norm_{name}' for name in beta_names]
        df = pd.DataFrame(result, columns=columns)

        results = []


        for column in df.columns[1:]:
            min_val = df[column].min()
            min_indices = np.where(df[column] == min_val)[0]

            if len(min_indices) > 1 and np.all(np.diff(min_indices) == 1):

                # avg_index = np.mean(min_indices)
                avg_index = np.percentile(min_indices, 72.5)
                # 100 50 50
                # 90 100 100
                # 80 50 50
            else:

                avg_index = min_indices[0]

            selected_bn = np.interp(avg_index, np.arange(len(df)), df['bn'].to_numpy())
            results.append([min_val, selected_bn])
            results_array = np.array(results)

        for i, name in enumerate(variable_names):
            globals()[f'bn_{name}'] = results_array[i, 1]

        for name in variable_names:
            beta_var = globals()[f'beta_{name}']
            bn_var = globals()[f'bn_{name}']
            beta_var[np.abs(beta_var) <= bn_var] = 0

        for name in variable_names:
            sigma = compute_sigma(Y, X, globals()[f'beta_{name}'])
            globals()[f'sigma_{name}'] = sigma

        Sigma_landweber += np.abs(sigma_Landweber**2 - std_dev**2)
        Sigma_landweber_debias += np.abs(sigma_Landweber_debias**2 - std_dev**2)
        Norm_landweber += np.linalg.norm(beta_Landweber-beta)
        Norm_landweber_debias += np.linalg.norm(beta_Landweber_debias-beta)

        norminf_Landweber_list = []
        norminf_Landweber_debias_list = []
        results = bootstrap_errors(
            spectral_design, 'Landweber', selected_parameter, selection_settings,
            np.column_stack([beta_Landweber, beta_Landweber_debias]),
            [sigma_Landweber, sigma_Landweber_debias], [bn_Landweber, bn_Landweber_debias],
            B, seed_root, x, backend=backend, batch_size=batch_size,
            cpu_workers=cpu_workers, blas_threads=blas_threads, gpu_devices=gpu_devices,
        )
        for norminf_Landweber, norminf_Landweber_debias in results:
            norminf_Landweber_list.append(norminf_Landweber)
            norminf_Landweber_debias_list.append(norminf_Landweber_debias)
        inf_landweber = np.linalg.norm(beta - beta_Landweber, ord=np.inf)
        inf_landweber_debias = np.linalg.norm(beta - beta_Landweber_debias, ord=np.inf)
        Inf_norm_landweber += inf_landweber
        Inf_norm_landweber_debias += inf_landweber_debias
        if inf_landweber <= np.percentile(norminf_Landweber_list, 100 * confidence):
            i_landweber += 1
        if inf_landweber_debias <= np.percentile(norminf_Landweber_debias_list, 100 * confidence):
            i_landweber_debias += 1

        run.record(
            x, selected_parameter=selected_parameter,
            thresholds=[bn_Landweber, bn_Landweber_debias],
            estimates=np.column_stack([beta_Landweber, beta_Landweber_debias]),
            sigma=[sigma_Landweber, sigma_Landweber_debias],
            bootstrap_errors=np.asarray(results),
            threshold_scan=df.to_numpy(), threshold_columns=list(df.columns),
            diagnostics={**selection_diagnostics, 'parameter_kind': 'steps', 'selected_residual_norm': selected_residual_norm, 'max_steps': max_iter, 'at_iteration_cap': bool(k_Landweber == max_iter)},
        )


In [ ]:
# Read all committed replications, including those restored from a checkpoint.
summary = run.summary
i_landweber = int(round(summary['coverage_original'] * N))
i_landweber_debias = int(round(summary['coverage_debiased'] * N))
Sigma_landweber = summary['variance_absolute_error_original'] * N
Sigma_landweber_debias = summary['variance_absolute_error_debiased'] * N
Inf_norm_landweber = summary['linf_original'] * N
Inf_norm_landweber_debias = summary['linf_debiased'] * N
Norm_landweber = summary['l2_original'] * N
Norm_landweber_debias = summary['l2_debiased'] * N
print(pd.Series(summary, name='Landweber'))
print('Saved results:', run.directory)
